In [1]:
import argparse
import os
import sys
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score # For clustering

import plotly.express as px

import json

In [2]:
INPUT_DIR = "../../data/embedded"
OUTPUT_DIR = "../../data/results"
GENERATE_PLOT = True
ALGORITHM = "dbscan"
N_CLUSTERS = 5
EPS = 10.0
MIN_SAMPLES = 10
REDUCER = "pca"
PERPLEXITY = 30.0

In [3]:
def load_data_recursive(input_dir):
    embeddings_list = []
    filenames = []

    print(f"Scanning '{input_dir}' for embeddings...")

    files_found = 0
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".npy"):
                file_path = os.path.join(root, file)

                try:
                    data = np.load(file_path)

                    if data.ndim == 0:
                        continue
                    elif data.ndim == 2:
                        data = data.squeeze()

                    if data.shape != (1024,):
                        continue

                    embeddings_list.append(data)
                    filenames.append(file.replace(".npy", ""))
                    files_found += 1

                except Exception as e:
                    print(f"Error loading {file}: {e}")

    if files_found == 0:
        return None, None

    print(f"Found {files_found} valid files.")

    X = np.vstack(embeddings_list)
    return X, filenames

In [ ]:
# LOAD DATA
if not os.path.exists(INPUT_DIR):
    print(f"Error: Directory '{INPUT_DIR}' not found.")
    sys.exit(0)

X, file_labels = load_data_recursive(INPUT_DIR)

if X is None:
    print("No valid embeddings found. Check your directory.")
    sys.exit(0)

print(f"Final Matrix Shape: {X.shape} (Samples: {X.shape[0]}, Features: {X.shape[1]})")

In [ ]:
# CREATE OUTPUT DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# PREPROCESSING
print("Scaling features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# CLUSTERING KMEANS
def cluster_kmeans(data, n_clusters):
    print(f"Running Clustering (kmeans)...")
    print(f"KMeans with {n_clusters} clusters")
    
    model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    
    return model.fit_predict(data)

In [ ]:
# CLUSTERING DBSCAN
def cluster_dbscan(data, eps, min_samples):
    print(f"Running Clustering (dbscan)...")
    print(f"DBSCAN with eps={eps}, min_samples={min_samples}")
    
    model = DBSCAN(eps=eps, min_samples=min_samples)
    
    return model.fit_predict(data)

In [ ]:
# DIMENSIONALITY REDUCTION
def reduce_pca(data, n_components):
    print(f"Reducing dimensions using (PCA)...")

    reducer = PCA(n_components=n_components)
    return reducer.fit_transform(data)

In [ ]:
def reduce_tsne(data, perplexity, n_components):
    pca_50 = PCA(n_components=min(50, X.shape[1]))
    X_pca = pca_50.fit_transform(data)

    tsne = TSNE(n_components=n_components, perplexity=perplexity, random_state=42, init='pca', learning_rate='auto')
    return tsne.fit_transform(X_pca)

In [ ]:
# VISUALIZATION
def create_save_scatter_plot(data, labels, file_labels, true_labels_path = '../../data/preprocessed/cleaned_labels.csv', title = "MOMENT Embeddings: PCA/TSNE Projection<br>(KMEANS/DBSCAN)", output_dir = "../../data/resultsplot.html):
    df_plot = pd.DataFrame(data, columns=['Component 1', 'Component 2'])
    
    df_plot['Cluster'] = labels.astype(str) 
    df_plot['Source'] = file_labels
    
    df_plot['ecg_id'] = df_plot['Source'].astype(int)
    
    try:
        true_labels_df = pd.read_csv(true_labels_path)
        df_plot = df_plot.merge(true_labels_df[['ecg_id', 'label']], on='ecg_id', how='left')
        df_plot['label'] = df_plot['label'].fillna('Unknown')
    except Exception as e:
        print(f"Could not load true labels: {e}")
        df_plot['label'] = 'Unknown'
    
    print("Generating Interactive Plot...")

    fig = px.scatter(
        df_plot,
        x='Component 1',
        y='Component 2',
        color='label',
        symbol='Cluster',
        hover_name='Source',
        hover_data={'label': True, 'Cluster': True, 'Component 1': False, 'Component 2': False},
        title=title,
        labels={
            'Component 1': "PCA/TSNE Dimension 1",
            'Component 2': "PCA/TSNE Dimension 2"
        },
        color_discrete_sequence=px.colors.qualitative.Plotly
    )

    fig.update_traces(marker=dict(size=10, line=dict(width=1, color='DarkSlateGrey')), opacity=0.8)
    fig.update_layout(template="plotly_white")

    fig.write_html(output_dir)
    print(f"Interactive plot saved to: {output_dir}")

In [ ]:
# DATA JSON
print("Exporting raw data for GUI...")

json_save_path = os.path.join(args['output_dir'], "real_chart_data.json")
chart_data = []

for idx, row in df_plot.iterrows():
    chart_data.append({
        "x": row['Component 1'],
        "y": row['Component 2'],
        "cluster": row['Cluster'],
        "source": row['Source'],
        "label": row['label']
    })

with open(json_save_path, 'w') as f:
    json.dump(chart_data, f, indent=4)

print(f"GUI data saved to: {json_save_path}")

In [ ]:
# CLUSTERING EVALUATION
print("\n" + "="*40)
print("CLUSTERING METRICS")
print("="*40)

# Silhouette Score
sil_score = silhouette_score(X_scaled, labels)
print(f"Silhouette Score: {sil_score:.4f} (Closer to 1 = better separated)")

# External Metrics
valid_idx = df_plot['label'] != 'Unknown'
true_labels_valid = df_plot.loc[valid_idx, 'label']
pred_clusters_valid = df_plot.loc[valid_idx, 'Cluster']

if len(true_labels_valid) > 0:
    # Calculate ARI and NMI
    ari = adjusted_rand_score(true_labels_valid, pred_clusters_valid)
    nmi = normalized_mutual_info_score(true_labels_valid, pred_clusters_valid)
    
    print(f"Adjusted Rand Index (ARI): {ari:.4f} (Closer to 1 = better match to true labels)")
    print(f"Normalized Mutual Info (NMI):  {nmi:.4f} (Closer to 1 = more information shared)")
else:
    print("Could not calculate ARI/NMI: No true labels found.")

print("="*40 + "\n")